In [1]:
import numpy as np
import pandas as pd
import pickle
import scipy
import time
import warnings
from collections import namedtuple

In [2]:
# define data format
from collections import namedtuple
RNAseqData = namedtuple('RNAseqData', 'counts genes cells clusterIDs clusterLabels clustergroupIDs '
                                      'clusterColors clusterNames clustergroupNames cellnames mappingrates')

In [3]:
# extract data from a excel file.
patch_data = pd.read_excel('../patchSeq/data/Ins_patchSeq.xlsx',index_col='cellName')
patch_data

,gene_order,s173,s174,s175,s177,s209,s210,s211,s212,s213,...,n87,n88,n89,n8,n90,n91,n92,n94,n96,n9
cellName,,,,,,,,,,,,,,,,,,,,,
cellType,"""""",VEN,VEN,PC,NaN,INTER,PC,VEN,PC,NaN,...,NaN,INTER,INTER,NaN,VEN,INTER,NaN,INTER,INTER,NaN
cellSubtype,"""""",VEN_Long,VEN_Long,PC_L5,NaN,INTER_L5,PC_L6,VEN_Long,PC_L5,NaN,...,NaN,INTER_L5,INTER_L5,NaN,VEN_Short,L23_INTER,NaN,L23_INTER,L23_INTER,NaN
MappingRate,"""""",0.8957,0.7971,0.8867,0.8785,0.8207,0.7865,0.8871,0.8337,0.8561,...,0.8258,0.865,0.7652,0.76,0.8734,0.8053,0.8213,0.8197,0.751,0.2173
batch,"""""",batch-1,batch-1,batch-1,batch-1,batch-1,batch-1,batch-1,batch-1,batch-1,...,batch-5,batch-5,batch-5,batch-5,batch-5,batch-5,batch-5,batch-5,batch-5,batch-5
# of Total Reads,"""""",2286636.72,7547298.05,4096627.48,9263391.89,4834936.14,6305879.98,4812780.15,5636087.57,4163858.02,...,2101502.04,1740485.41,3306850.15,1721003.1,569116.48,3291647.51,2750171.7,4211850.26,1653226.49,1322817.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZXDC,16456,0,0,4,0,0,0,0,0,13,...,0,0,0,0,0,0,0,0,0,0
ZYG11A,16457,0,0,0,0,18,0,0,50,4,...,0,0,3,0,0,0,30,0,0,0
ZYG11B,16458,1,15,36,213,160,259,328,124,93,...,92,522,283,33,3,0,441,105,6,0


In [4]:
patch_data = patch_data.drop('gene_order',axis=1)
genes_patch = np.array(patch_data.index[5:])

# exclude the cells with mapping rates < 0.4
patch_data=patch_data.loc[:,patch_data.iloc[2,:] >= 0.4]

counts_patch = patch_data.values[5:,:].astype('float').T
cells_patch = patch_data.columns
cell_type = patch_data.values[0,:]
cell_subType = patch_data.values[1,:]
mapRate_str = patch_data.values[2,:]
mappingrates=mapRate_str
numTotalReads = patch_data.values[3,:]

In [5]:
# resort the cell properties for loading into a pickle file.
patchCell_groups=np.ones(cell_type.size)
for i in range(cell_type.size):
    if cell_type[i] == 'PC':
        patchCell_groups[i]=1
    elif ((cell_type[i] == 'VEN') | (cell_type[i] == 'Long') | (cell_type[i] == 'Short') ):
        patchCell_groups[i]=2
    elif ((cell_type[i] == 'Inter') | (cell_type[i] == 'inter') | (cell_type[i] == 'INTER')):
        patchCell_groups[i]=0
    elif ( np.isnan(cell_type[i]) | (cell_type[i] == 'NAN') | (cell_type[i] == 'Nan') ):
        patchCell_groups[i]=-1 
        
clusterIDs=np.zeros(cells_patch.size)  # the default cell subtype id is 0 (nan, none defiend)
clusterIDs[cell_subType=='INTER_L23']=1
clusterIDs[cell_subType=='L23_INTER']=1
clusterIDs[cell_subType=='INTER_L1']=1
clusterIDs[cell_subType=='INTER_L5']=1
clusterIDs[cell_subType=='INTER_L6']=1
clusterIDs[cell_subType=='PC_L23']=2
clusterIDs[cell_subType=='L23_PC']=2
clusterIDs[cell_subType=='PC_L5']=3
clusterIDs[cell_subType=='L5_PC']=3
clusterIDs[cell_subType=='VEN_Long']=4
clusterIDs[cell_subType=='VEN_Short']=5
clusterIDs[cell_subType=='VEN']=6
clusterIDs[cell_subType=='PC_L6']=7
clusterIDs[cell_subType=='L6_PC']=7
    
# clusterColors=np.array(['#DDACC9', '#A700FF', '#FF00B3'])

patchSeqData = RNAseqData(counts=counts_patch, genes=genes_patch, cells=cells_patch, clusterIDs=clusterIDs, 
                          clusterLabels=None, clustergroupIDs=patchCell_groups, clusterColors = None, 
                          clusterNames = cell_subType, clustergroupNames = None,cellnames=cells_patch,
                          mappingrates=mappingrates)
                        
pickle.dump(patchSeqData, open('./data/processedData/insPatchDataNew251010.pickle', 'wb'))  

# END OF CODES